In [19]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import sys

# Ensure openpyxl is available (required for Excel export)
try:
    import openpyxl
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl'])
    import openpyxl

# Clear any existing handlers to prevent duplicates
root_logger = logging.getLogger()
if root_logger.handlers:
    for handler in root_logger.handlers:
        root_logger.removeHandler(handler)

# Configure logging FIRST before importing DataLoader
logging.basicConfig(level=logging.WARNING, force=True)

# Suppress specific loggers that are verbose
for logger_name in ['__main__', 'snowflake', 'azure', 'urllib3']:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# Add the project root to the path to import custom modules
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import the DataLoader class
from scripts.data_preparation.load import DataLoader

print("✓ Libraries imported successfully")
print(f"✓ openpyxl {openpyxl.__version__} available")
print("✓ Logging configured (INFO messages suppressed)")


✓ Libraries imported successfully
✓ openpyxl 3.1.5 available
✓ Logging configured (INFO messages suppressed)


In [20]:
# Getting email data from startdate to enddate
startdate = '2026-05-01'
enddate = '2026-05-31' # '2026-05-11'

print(f"Tracking Email History will be Retrieved from {startdate} to {enddate}")

Tracking Email History will be Retrieved from 2026-05-01 to 2026-05-31


In [21]:
# Force reload the DataLoader module to ensure latest code is used
import importlib
import sys

# Remove from cache if present, then reimport fresh
if 'scripts.data_preparation.load' in sys.modules:
    del sys.modules['scripts.data_preparation.load']

from scripts.data_preparation.load import DataLoader
print("✓ DataLoader module loaded fresh")

# Verify the parameter substitution method being used
import inspect
src = inspect.getsource(DataLoader.load_from_snowflake)
if 'query.replace' in src:
    print("✓ Using str.replace() for parameter substitution (correct)")
elif 'query.format' in src:
    print("⚠️  Still using str.format() - reload did not pick up changes!")


✓ DataLoader module loaded fresh
⚠️  Still using str.format() - reload did not pick up changes!


## 1. Load Tracking CheckCall Data from Snowflake

Load the tracking checkcall data from Snowflake using the DataLoader class.

In [22]:
# Execute the query to get historical load data
print("Querying Snowflake for Tracking checkcall data...")
print(f"Date Range: {startdate} to {enddate}")

loader = DataLoader()

df = loader.load_from_snowflake(
    sql_file_path='../sql/loadTrackingOverprocessing3.sql', # Use this sql file, it has the most updated and correct logic
    params={'startdate': startdate, 'enddate': enddate}
)

# Convert datetime columns immediately after load.
# All three are cast to ::TIMESTAMP_NTZ::string in SQL, which strips the timezone
# offset while preserving the Chicago local time value — no utc=True needed.
df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
df['OG_APPTOPEN']          = pd.to_datetime(df['OG_APPTOPEN'], errors='coerce')
df['OG_SCHEDDATETIME']     = pd.to_datetime(df['OG_SCHEDDATETIME'], errors='coerce')

print(f"\n✓ Loaded {len(df):,} rows from Snowflake")
print(f"  Date range in data: {df['ENTERED_DATETIME_CST'].min()} to {df['ENTERED_DATETIME_CST'].max()}")
print(f"  Columns: {len(df.columns)}")
df.head()


Querying Snowflake for Tracking checkcall data...
Date Range: 2026-05-01 to 2026-05-31


/home/azureuser/loadTrackingOverProcessing/LoadTrackingOverProcessing/scripts/data_preparation/load.py:169: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.snowflake_conn)



✓ Loaded 44,764,078 rows from Snowflake
  Date range in data: 1961-05-06 21:00:00 to 2042-05-14 08:00:00
  Columns: 26


,LOAD_NUM,CUSTOMERCODE,EAR2ID,EAR2NAME,CHECK_CALL_TYPE,DESCRIPTION,ENTERED_DATETIME_CST,CITY,STATE,COUNTRY,...,JOB_FAMILY_DESCRIPTION,ROLLUPBRANCHNAME,BRANCHSUBREGION2,HUMAN_ENTERED_CHECKCALL_FLAG,TRACKING_IDENTIFIER_CLEAN,TRACKING_IDENTIFIER_TYPE,TRACKING_METHOD_TYPE,APPT_TYPE,OG_SCHEDDATETIME,OG_APPTOPEN
0,552443763,None,NaN,None,D-Open,W6436258,1961-05-06 21:00:00,WOODSTOCK,IL,United States,...,None,None,None,NaN,None,None,None,APPOINTMENTS SET,NaT,NaT
1,551886572,None,NaN,None,D-Open,W95483958,2008-05-07 07:00:00,BARABOO,WI,United States,...,None,None,None,NaN,None,None,None,APPOINTMENTS SET,NaT,NaT
2,552617531,None,NaN,None,D-Close,W95859476,2011-05-01 15:00:00,LANSING,MI,United States,...,None,None,None,NaN,None,None,None,RESCHEDULES SET,NaT,NaT
3,552573880,None,NaN,None,D-Open,W79621427,2016-05-08 07:00:00,Durham,NC,United States,...,None,None,None,NaN,None,None,None,APPOINTMENTS SET,NaT,NaT
4,553714817,None,NaN,None,D-Open,W57413171,2022-02-05 07:00:00,North Attleboro,MA,United States,...,None,None,None,NaN,None,None,None,RESCHEDULES SET,NaT,NaT


In [23]:
# df[(df['LOAD_NUM']==546748574) & (pd.isnull(df['JOB_FAMILY_DESCRIPTION'])==False)][['LOAD_NUM','EAR2NAME','DESCRIPTION','ENTERED_DATETIME_CST','OG_APPTOPEN','OG_SCHEDDATETIME','UPDATE_USER']]


In [24]:
# df[(df['LOAD_NUM']==548544546)][['LOAD_NUM','EAR2NAME','DESCRIPTION','ENTERED_DATETIME_CST','OG_APPTOPEN','OG_SCHEDDATETIME','UPDATE_USER','AUTOMATED']] # & (pd.isnull(df['JOB_FAMILY_DESCRIPTION'])==False)][['LOAD_NUM','EAR2NAME','DESCRIPTION','ENTERED_DATETIME_CST','OG_APPTOPEN','OG_SCHEDDATETIME','UPDATE_USER']]
# # # # 546748574

In [25]:
# df[df['LOAD_NUM']==552545690] # 554287901

In [26]:
# df.columns

In [27]:
def get_loads_with_manual_cc_before_pickup(df, early_threshold_hours=4):
    """
    Identifies LOAD_NUMs that have at least one manual check call
    (CHECK_CALL_TYPE = 'CC' and AUTOMATED = False) recorded before
    the original pickup open appointment (OG_APPTOPEN).

    Uses OG_APPTOPEN rather than the ENTERED_DATETIME_CST of the first P-Open
    event, because the appointment time in effect at the time of the check call
    may differ from when the P-Open was eventually recorded (e.g. after reschedules).
    OG_APPTOPEN is the appointment open time that was active when the CC was entered.
    Note: a single load may have multiple different OG_APPTOPEN values across its
    manual CCs as appointments are rescheduled, so it is not included in the output.

    Excludes check calls where UPDATE_USER = 'DATASCIENCE'.
    Only includes check calls where HUMAN_ENTERED_CHECKCALL_FLAG = 1.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake containing columns:
                           LOAD_NUM, CHECK_CALL_TYPE, AUTOMATED, ENTERED_DATETIME_CST,
                           UPDATE_USER, HUMAN_ENTERED_CHECKCALL_FLAG, OG_APPTOPEN
        early_threshold_hours (int): Hours before OG_APPTOPEN to classify as "very early". Default 4.

    Returns:
        pd.DataFrame: DataFrame of qualifying LOAD_NUMs with supporting details:
                      - LOAD_NUM
                      - manual_cc_before_pickup_count: number of manual CCs before OG_APPTOPEN
                      - earliest_manual_cc: datetime of the first manual CC before OG_APPTOPEN
                      - manual_cc_early_count: manual CCs more than `early_threshold_hours`
                                               before OG_APPTOPEN (very premature touches)
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
    df['OG_APPTOPEN'] = pd.to_datetime(df['OG_APPTOPEN'], errors='coerce')

    # Get manual check calls (CC + not automated + not DATASCIENCE user + human entered)
    # Include OG_APPTOPEN — the appointment open time active at the time of each CC
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1) &
        (df['OG_APPTOPEN'].notna())
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST', 'OG_APPTOPEN']].copy()

    # Keep only manual CCs that occurred BEFORE their row's OG_APPTOPEN
    manual_cc_before = manual_cc[
        manual_cc['ENTERED_DATETIME_CST'] < manual_cc['OG_APPTOPEN']
    ].copy()

    # Flag CCs that are more than `early_threshold_hours` before OG_APPTOPEN
    early_cutoff = pd.Timedelta(hours=early_threshold_hours)
    manual_cc_before['is_very_early'] = (
        manual_cc_before['OG_APPTOPEN'] - manual_cc_before['ENTERED_DATETIME_CST']
    ) > early_cutoff

    # Aggregate per load
    result = (
        manual_cc_before.groupby('LOAD_NUM')
        .agg(
            manual_cc_before_pickup_count=('ENTERED_DATETIME_CST', 'count'),
            earliest_manual_cc=('ENTERED_DATETIME_CST', 'min'),
            manual_cc_early_count=('is_very_early', 'sum')
        )
        .reset_index()
    )

    result['manual_cc_early_count'] = result['manual_cc_early_count'].astype(int)
    result = result.sort_values('manual_cc_before_pickup_count', ascending=False)

    return result


# Run the function
total_loads = df['LOAD_NUM'].nunique()
loads_with_early_manual_cc = get_loads_with_manual_cc_before_pickup(df, early_threshold_hours=4)
qualifying_loads = len(loads_with_early_manual_cc)
total_before_pickup = loads_with_early_manual_cc['manual_cc_before_pickup_count'].sum()
total_very_early = loads_with_early_manual_cc['manual_cc_early_count'].sum()

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"LOAD_NUMs with manual CC before OG_APPTOPEN:       {qualifying_loads:,}")
print(f"Percentage:                                        {qualifying_loads / total_loads * 100:.1f}%")
print(f"Total manual CCs before OG_APPTOPEN:               {total_before_pickup:,}")
print(f"  of which >4 hrs before OG_APPTOPEN:              {total_very_early:,} ({total_very_early / total_before_pickup * 100:.1f}%)")
loads_with_early_manual_cc.head(10)


Total distinct LOAD_NUMs in dataset:               311,359
LOAD_NUMs with manual CC before OG_APPTOPEN:       11,458
Percentage:                                        3.7%
Total manual CCs before OG_APPTOPEN:               12,436
  of which >4 hrs before OG_APPTOPEN:              2,481 (20.0%)


,LOAD_NUM,manual_cc_before_pickup_count,earliest_manual_cc,manual_cc_early_count
11357,555191784,8,2026-05-28 17:43:00,8
6267,553044827,7,2026-05-11 15:42:00,7
40,548544546,6,2026-05-05 17:48:00,6
10126,554287901,5,2026-05-21 19:54:00,4
8812,553780378,4,2026-05-21 13:16:00,1
5740,552902148,4,2026-05-13 07:32:00,4
5276,552817819,4,2026-05-14 09:45:00,1
8620,553742310,4,2026-05-18 15:17:00,1
2441,552030189,4,2026-05-12 13:59:00,1
1806,551828882,4,2026-05-06 17:24:00,1


In [28]:
def get_early_manual_cc_summary(df, early_threshold_hours=4):
    """
    Summarizes manual check calls that occurred more than `early_threshold_hours`
    before the original pickup open appointment (OG_APPTOPEN) at three aggregated
    levels, plus a raw row-level detail table:
      1. EAR2NAME — to identify which customer segments drive the most early touches
      2. UPDATE_USER + ROLLUPBRANCHNAME — to identify which dispatcher/branch combos
         are entering the most premature manual CCs
      3. UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME — to identify which user/branch/customer
         segment combos are driving the most premature manual CCs
      4. Raw detail — one row per qualifying manual CC with load/user/customer context

    Uses the same manual CC filter as get_loads_with_manual_cc_before_pickup:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1
        - OG_APPTOPEN is not null
        - ENTERED_DATETIME_CST < OG_APPTOPEN
        - OG_APPTOPEN - ENTERED_DATETIME_CST > early_threshold_hours

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        early_threshold_hours (int): Hours threshold. Default 4.

    Returns:
        tuple:
            - by_ear2 (pd.DataFrame): Aggregated by EAR2NAME
            - by_user_branch (pd.DataFrame): Aggregated by UPDATE_USER + ROLLUPBRANCHNAME
            - by_user_branch_ear2 (pd.DataFrame): Aggregated by UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME
            - detail (pd.DataFrame): Row-level dump of all qualifying manual CCs with
                                     LOAD_NUM, ENTERED_DATETIME_CST, UPDATE_USER,
                                     CUSTOMERCODE, EAR2NAME, EAR2ID
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
    df['OG_APPTOPEN'] = pd.to_datetime(df['OG_APPTOPEN'], errors='coerce')
    early_cutoff = pd.Timedelta(hours=early_threshold_hours)

    # Column name reflects the threshold so the viewer knows what it means
    early_col = f'manual_cc_{early_threshold_hours}hr_before_og_apptopen'

    # Filter to manual CCs with a valid OG_APPTOPEN
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1) &
        (df['OG_APPTOPEN'].notna())
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST', 'OG_APPTOPEN', 'UPDATE_USER',
       'EAR2NAME', 'EAR2ID', 'CUSTOMERCODE', 'ROLLUPBRANCHNAME']].copy()

    # Keep only CCs more than `early_threshold_hours` before OG_APPTOPEN
    very_early = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] < manual_cc['OG_APPTOPEN']) &
        ((manual_cc['OG_APPTOPEN'] - manual_cc['ENTERED_DATETIME_CST']) > early_cutoff)
    ].copy()

    # --- Summary 1: By EAR2NAME ---
    by_ear2 = (
        very_early.groupby('EAR2NAME')
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            **{early_col: ('ENTERED_DATETIME_CST', 'count')}
        )
        .reset_index()
        .sort_values(early_col, ascending=False)
    )

    # --- Summary 2: By UPDATE_USER + ROLLUPBRANCHNAME ---
    by_user_branch = (
        very_early.groupby(['UPDATE_USER', 'ROLLUPBRANCHNAME'])
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            **{early_col: ('ENTERED_DATETIME_CST', 'count')}
        )
        .reset_index()
        .sort_values(early_col, ascending=False)
    )

    # --- Summary 3: By UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME ---
    by_user_branch_ear2 = (
        very_early.groupby(['UPDATE_USER', 'ROLLUPBRANCHNAME', 'EAR2NAME'])
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            **{early_col: ('ENTERED_DATETIME_CST', 'count')}
        )
        .reset_index()
        .sort_values(early_col, ascending=False)
    )

    # --- Detail: Row-level dump of all qualifying manual CCs ---
    detail = (
        very_early[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'UPDATE_USER',
                    'CUSTOMERCODE', 'EAR2NAME', 'EAR2ID']]
        .sort_values(['LOAD_NUM', 'ENTERED_DATETIME_CST'])
        .reset_index(drop=True)
    )

    return by_ear2, by_user_branch, by_user_branch_ear2, detail


# Run the function
early_by_ear2, early_by_user_branch, early_by_user_branch_ear2, early_detail = get_early_manual_cc_summary(df, early_threshold_hours=4)

print("=== Manual CCs >4 hrs before OG_APPTOPEN by EAR2NAME ===")
print(f"Distinct EAR2s with early touches: {len(early_by_ear2):,}")
print()
display(early_by_ear2.head(15))

print("\n=== Manual CCs >4 hrs before OG_APPTOPEN by UPDATE_USER / ROLLUPBRANCHNAME ===")
print(f"Distinct user/branch combos with early touches: {len(early_by_user_branch):,}")
print()
display(early_by_user_branch.head(20))

print("\n=== Manual CCs >4 hrs before OG_APPTOPEN by UPDATE_USER / ROLLUPBRANCHNAME / EAR2NAME ===")
print(f"Distinct user/branch/EAR2 combos with early touches: {len(early_by_user_branch_ear2):,}")
print()
display(early_by_user_branch_ear2.head(20))

print("\n=== Raw Manual CC Detail (>4 hrs before OG_APPTOPEN) ===")
print(f"Total qualifying manual CC rows: {len(early_detail):,}")
print()
display(early_detail.head(20))


=== Manual CCs >4 hrs before OG_APPTOPEN by EAR2NAME ===
Distinct EAR2s with early touches: 631



,EAR2NAME,distinct_loads,manual_cc_4hr_before_og_apptopen
532,The Coca-Cola Company,106,110
459,Robinson Fresh,92,95
280,International Paper Company,72,83
137,"Constellation Brands, Inc.",42,46
202,"Floor And Decor Outlets Of America, Inc.",35,41
548,The Quaker Oats Company,34,37
247,Harvest Hill Beverage Company,34,36
362,Molson Coors Beverage Company,31,32
269,"INDEPENDENT PURCHASING COOPERATIVE, INC.",25,30
517,Target Corporation,27,27



=== Manual CCs >4 hrs before OG_APPTOPEN by UPDATE_USER / ROLLUPBRANCHNAME ===
Distinct user/branch combos with early touches: 317



,UPDATE_USER,ROLLUPBRANCHNAME,distinct_loads,manual_cc_4hr_before_og_apptopen
35,CAMPJOS,Memphis Capacity,345,406
128,HOVAGAV,Chicago Central Capacity,117,123
228,RADDCHR,Michigan Capacity,114,117
249,SALAADR,TC Capacity,86,86
62,CUBIGLO,Chicago Central Capacity,74,80
42,CASTAN,Dallas Capacity,77,79
271,SPAEDYL,NAST Flatbed,70,70
262,SHIMGAB,Kansas City Capacity,64,64
243,RODRTHO,LA Capacity,44,46
292,TORRMAG,Mexico Border Capacity,41,43



=== Manual CCs >4 hrs before OG_APPTOPEN by UPDATE_USER / ROLLUPBRANCHNAME / EAR2NAME ===
Distinct user/branch/EAR2 combos with early touches: 1,595



,UPDATE_USER,ROLLUPBRANCHNAME,EAR2NAME,distinct_loads,manual_cc_4hr_before_og_apptopen
269,CAMPJOS,Memphis Capacity,International Paper Company,58,68
1309,SALAADR,TC Capacity,Robinson Fresh,60,60
244,CAMPJOS,Memphis Capacity,"Floor And Decor Outlets Of America, Inc.",34,40
230,CAMPJOS,Memphis Capacity,"Constellation Brands, Inc.",21,25
1447,SPAEDYL,NAST Flatbed,Henry Company LLC,19,19
316,CAMPJOS,Memphis Capacity,The Coca-Cola Company,17,19
482,CUBIGLO,Chicago Central Capacity,The Coca-Cola Company,11,12
495,DEHOROB,Dallas Capacity,"Bluetriton Brands, Inc.",12,12
1457,SPAEDYL,NAST Flatbed,Pacific Gas and Electric Company,11,11
1459,SPAEDYL,NAST Flatbed,Pool Corporation,10,10



=== Raw Manual CC Detail (>4 hrs before OG_APPTOPEN) ===
Total qualifying manual CC rows: 2,481



,LOAD_NUM,ENTERED_DATETIME_CST,UPDATE_USER,CUSTOMERCODE,EAR2NAME,EAR2ID
0,546316972,2026-05-26 08:03:00,WHITRYAN,C3013029,"Fuji Component Parts USA, Inc.",427564.0
1,546334854,2026-05-04 23:05:00,CAMPJOS,C655645,"Floor And Decor Outlets Of America, Inc.",1025.0
2,546334854,2026-05-04 23:06:00,CAMPJOS,C655645,"Floor And Decor Outlets Of America, Inc.",1025.0
3,546334858,2026-05-04 23:06:00,CAMPJOS,C655645,"Floor And Decor Outlets Of America, Inc.",1025.0
4,546610486,2026-05-15 09:37:00,GRANHUN,C5166081,Omya Inc.,26222.0
5,547472408,2026-05-20 10:40:00,ASANKEV,C8047109,Harvest Hill Beverage Company,552704.0
6,547483074,2026-05-04 09:37:00,HOVAGAV,C7357050,"Pepperidge Farm, Incorporated",1018864.0
7,548071174,2026-05-05 14:30:00,MILLIWO,C8928390,Planet Harvest,1148337.0
8,548174299,2026-05-14 08:38:00,CAMPJOS,C8739066,"Diamond Crystal Brands, Inc.",491.0
9,548176730,2026-05-14 08:39:00,CAMPJOS,C8739066,"Diamond Crystal Brands, Inc.",491.0


In [29]:
print("Total manual CCs >4 hrs before OG_APPTOPEN (EAR2 summary):                    ", early_by_ear2['manual_cc_4hr_before_og_apptopen'].sum())
print("Total manual CCs >4 hrs before OG_APPTOPEN (user/branch summary):            ", early_by_user_branch['manual_cc_4hr_before_og_apptopen'].sum())
print("Total manual CCs >4 hrs before OG_APPTOPEN (user/branch/EAR2 summary):      ", early_by_user_branch_ear2['manual_cc_4hr_before_og_apptopen'].sum())
print("Total manual CCs >4 hrs before OG_APPTOPEN (detail rows):                    ", len(early_detail))


Total manual CCs >4 hrs before OG_APPTOPEN (EAR2 summary):                     2474
Total manual CCs >4 hrs before OG_APPTOPEN (user/branch summary):             2481
Total manual CCs >4 hrs before OG_APPTOPEN (user/branch/EAR2 summary):       2474
Total manual CCs >4 hrs before OG_APPTOPEN (detail rows):                     2481


In [30]:
def get_load_nums_for_user_early_cc(df, update_user, early_threshold_hours=4):
    """
    Returns a list of LOAD_NUMs where the specified UPDATE_USER entered at least
    one manual check call more than `early_threshold_hours` before OG_APPTOPEN.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        update_user (str): The UPDATE_USER value to filter on.
        early_threshold_hours (int): Hours before OG_APPTOPEN threshold. Default 4.

    Returns:
        list: Sorted list of distinct LOAD_NUMs meeting the criteria.
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
    df['OG_APPTOPEN'] = pd.to_datetime(df['OG_APPTOPEN'], errors='coerce')
    early_cutoff = pd.Timedelta(hours=early_threshold_hours)

    mask = (
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] == update_user) &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1) &
        (df['OG_APPTOPEN'].notna()) &
        (df['ENTERED_DATETIME_CST'] < df['OG_APPTOPEN']) &
        ((df['OG_APPTOPEN'] - df['ENTERED_DATETIME_CST']) > early_cutoff)
    )

    load_nums = sorted(df.loc[mask, 'LOAD_NUM'].unique().tolist())
    print(f"UPDATE_USER:          '{update_user}'")
    print(f"Threshold:            >{early_threshold_hours} hrs before OG_APPTOPEN")
    print(f"Qualifying LOAD_NUMs: {len(load_nums):,}")
    return load_nums


# Example usage — replace with the user you want to investigate
# example_user = early_by_user_branch.iloc[0]['UPDATE_USER']
example_user = 'CAMPJOS'
user_load_nums = get_load_nums_for_user_early_cc(df, update_user=example_user, early_threshold_hours=4)
print(f"\nFirst 20 LOAD_NUMs: {user_load_nums[:10]}")


UPDATE_USER:          'CAMPJOS'
Threshold:            >4 hrs before OG_APPTOPEN
Qualifying LOAD_NUMs: 345

First 20 LOAD_NUMs: [546334854, 546334858, 548174299, 548176730, 548285578, 548285583, 548285604, 548285608, 548285614, 548285625]


In [31]:
# df[df['LOAD_NUM']==546334854][['LOAD_NUM','DESCRIPTION','ENTERED_DATETIME_CST','UPDATE_USER','OG_SCHEDDATETIME','OG_APPTOPEN']]

In [32]:
# # Export early manual CC summaries to Excel
# output_path = f'../data/processed/early_manual_cc_summary_{startdate}_to_{enddate}_updated.xlsx'

# with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
#     early_by_ear2.to_excel(writer, sheet_name='By_EAR2NAME', index=False)
#     early_by_user_branch.to_excel(writer, sheet_name='By_User_Branch', index=False)
#     early_by_user_branch_ear2.to_excel(writer, sheet_name='By_User_Branch_EAR2', index=False)
#     early_detail.to_excel(writer, sheet_name='Detail', index=False)

# print(f"✓ Excel file saved to: {output_path}")
# print(f"  Tab 'By_EAR2NAME':           {len(early_by_ear2):,} rows")
# print(f"  Tab 'By_User_Branch':        {len(early_by_user_branch):,} rows")
# print(f"  Tab 'By_User_Branch_EAR2':   {len(early_by_user_branch_ear2):,} rows")
# print(f"  Tab 'Detail':                {len(early_detail):,} rows")


In [33]:
# loads_with_early_manual_cc[loads_with_early_manual_cc['LOAD_NUM']==554287901]

In [34]:
print("Total Manual CCs before Pickup Open:", loads_with_early_manual_cc['manual_cc_before_pickup_count'].sum())

Total Manual CCs before Pickup Open: 12436


In [35]:
def get_manual_cc_near_auto_in_transit(df, window_minutes=30):
    """
    Identifies per load how many manual check calls occurred during active transit
    (between first P-Open and last D-Close) AND were entered within `window_minutes`
    AFTER an automated check call on the same load.

    The intent is to measure over-processing: manual CCs entered by a dispatcher
    shortly after automated tracking already fired — suggesting the dispatcher
    checked in unnecessarily when automation had just done so.

    Criteria for a manual check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1

    Criteria for an automated check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == True
        - HUMAN_ENTERED_CHECKCALL_FLAG == 0
        - UPDATE_USER != 'DATASCIENCE'

    Criteria for "redundant manual CC after auto":
        - At least one auto CC on the same load occurred within `window_minutes` BEFORE the manual CC

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        window_minutes (int): Look-back window in minutes. Default 30.

    Returns:
        pd.DataFrame: One row per load with:
            - LOAD_NUM
            - first_pickup_open: earliest P-Open datetime
            - last_dropoff_close: latest D-Close datetime
            - transit_manual_cc_total: total manual CCs in transit window
            - manual_cc_after_auto_count: manual CCs entered within window_minutes after an auto CC
            - pct_after_auto: percentage of transit manual CCs that followed an auto CC
            - avg_minutes_after_auto: average minutes between the closest preceding auto CC and
                                      the manual CC (qualifying manual CCs only)
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
    window = pd.Timedelta(minutes=window_minutes)

    # --- Transit window boundaries per load ---
    first_pickup = (
        df[df['CHECK_CALL_TYPE'] == 'P-Open']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .min()
        .rename('first_pickup_open')
    )
    last_dropoff = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
    )
    transit_bounds = pd.concat([first_pickup, last_dropoff], axis=1).dropna().reset_index()

    # --- Manual CCs ---
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Keep only manual CCs inside the transit window
    manual_cc = manual_cc.merge(transit_bounds, on='LOAD_NUM', how='inner')
    manual_cc = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] >= manual_cc['first_pickup_open']) &
        (manual_cc['ENTERED_DATETIME_CST'] <= manual_cc['last_dropoff_close'])
    ].copy()

    # --- Automated CCs ---
    auto_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == True) &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 0) &
        (df['UPDATE_USER'] != 'DATASCIENCE')
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].rename(
        columns={'ENTERED_DATETIME_CST': 'auto_datetime'}
    ).copy()

    # --- Cross-join per load and compute minutes from auto CC to manual CC ---
    # Positive value means the manual CC came AFTER the auto CC
    merged = manual_cc.merge(auto_cc, on='LOAD_NUM', how='left')
    merged['minutes_after_auto'] = (
        merged['ENTERED_DATETIME_CST'] - merged['auto_datetime']
    ).dt.total_seconds() / 60

    # Keep only pairings where auto CC preceded the manual CC within the window
    qualifying = merged[
        (merged['minutes_after_auto'] > 0) &
        (merged['minutes_after_auto'] <= window_minutes)
    ].copy()

    # For each manual CC, use the closest (minimum minutes) preceding auto CC
    closest_auto = (
        qualifying.groupby(['LOAD_NUM', 'ENTERED_DATETIME_CST'])['minutes_after_auto']
        .min()
        .reset_index()
        .rename(columns={'minutes_after_auto': 'min_minutes_after_auto'})
    )
    closest_auto['follows_auto'] = True

    # Join back to all manual CCs in transit
    manual_cc = manual_cc.merge(
        closest_auto[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'follows_auto', 'min_minutes_after_auto']],
        on=['LOAD_NUM', 'ENTERED_DATETIME_CST'],
        how='left'
    )
    manual_cc['follows_auto'] = manual_cc['follows_auto'].fillna(False)

    # --- Aggregate per load ---
    result = (
        manual_cc.groupby(['LOAD_NUM', 'first_pickup_open', 'last_dropoff_close'])
        .agg(
            transit_manual_cc_total=('ENTERED_DATETIME_CST', 'count'),
            manual_cc_after_auto_count=('follows_auto', 'sum'),
            avg_minutes_after_auto=('min_minutes_after_auto', 'mean')
        )
        .reset_index()
    )

    result['manual_cc_after_auto_count'] = result['manual_cc_after_auto_count'].astype(int)
    result['avg_minutes_after_auto'] = result['avg_minutes_after_auto'].round(1)
    result['pct_after_auto'] = (
        result['manual_cc_after_auto_count'] / result['transit_manual_cc_total'] * 100
    ).round(1)

    result = result.sort_values('manual_cc_after_auto_count', ascending=False)

    return result


# Run the function
in_transit_overprocessing = get_manual_cc_near_auto_in_transit(df, window_minutes=30)
qualifying_loads = len(in_transit_overprocessing)
total_after_auto = in_transit_overprocessing['manual_cc_after_auto_count'].sum()
total_transit_manual = in_transit_overprocessing['transit_manual_cc_total'].sum()
overall_avg_minutes = in_transit_overprocessing.loc[
    in_transit_overprocessing['manual_cc_after_auto_count'] > 0, 'avg_minutes_after_auto'
].mean()

print(f"Loads with manual CCs in transit window:           {qualifying_loads:,}")
print(f"Total manual CCs in transit window:                {total_transit_manual:,}")
print(f"Manual CCs within 30 min after auto CC:            {total_after_auto:,}")
print(f"Overall % after auto CC:                           {total_after_auto / total_transit_manual * 100:.1f}%")
print(f"Avg minutes after auto CC (loads with any):        {overall_avg_minutes:.1f} min")
in_transit_overprocessing.head(10)


Loads with manual CCs in transit window:           16,154
Total manual CCs in transit window:                21,162
Manual CCs within 30 min after auto CC:            9,894
Overall % after auto CC:                           46.8%
Avg minutes after auto CC (loads with any):        8.3 min


,LOAD_NUM,first_pickup_open,last_dropoff_close,transit_manual_cc_total,manual_cc_after_auto_count,avg_minutes_after_auto,pct_after_auto
922,551412317,2026-05-21 08:00:00,2026-05-27 09:00:00,19,19,9.7,100.0
12263,553916777,2026-05-21 08:00:00,2026-05-26 07:00:00,18,17,8.0,94.4
10056,553442682,2026-05-26 08:00:00,2026-06-01 05:30:00,20,17,6.6,85.0
1508,551733805,2026-05-01 08:00:00,2026-05-04 21:00:00,16,16,8.4,100.0
561,551098226,2026-05-01 08:00:00,2026-05-04 09:00:00,16,16,9.2,100.0
6591,552771412,2026-05-08 08:00:00,2026-05-12 15:00:00,15,15,5.1,100.0
4614,552424069,2026-05-05 06:00:00,2026-05-08 15:00:00,15,15,9.3,100.0
1504,551733783,2026-05-22 08:00:00,2026-05-26 11:00:00,15,15,4.6,100.0
11292,553660696,2026-05-18 07:00:00,2026-05-20 09:00:00,14,14,9.1,100.0
4491,552401308,2026-05-04 07:00:00,2026-05-06 09:00:00,15,14,11.2,93.3


In [37]:
def get_manual_cc_near_auto_in_transit_afterhours(df, window_minutes=30, branch_name='NAST After Hours'):
    """
    Same logic as get_manual_cc_near_auto_in_transit(), but restricts manual check calls
    to those entered by after-hours employees (ROLLUPBRANCHNAME == branch_name).

    Identifies per load how many after-hours manual CCs occurred during active transit
    (between first P-Open and last D-Close) AND were entered within `window_minutes`
    AFTER an automated check call on the same load.

    Criteria for an after-hours manual check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1
        - ROLLUPBRANCHNAME == branch_name

    Criteria for an automated check call (not filtered to after hours):
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == True
        - HUMAN_ENTERED_CHECKCALL_FLAG == 0
        - UPDATE_USER != 'DATASCIENCE'

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        window_minutes (int): Look-back window in minutes. Default 30.
        branch_name (str): ROLLUPBRANCHNAME to filter manual CCs to. Default 'NAST After Hours'.

    Returns:
        tuple:
            - load_summary (pd.DataFrame): One row per load with:
                  LOAD_NUM, first_pickup_open, last_dropoff_close,
                  transit_manual_cc_total, manual_cc_after_auto_count,
                  pct_after_auto, avg_minutes_after_auto
            - by_user (pd.DataFrame): Aggregated by UPDATE_USER with:
                  UPDATE_USER, distinct_loads, manual_cc_after_auto_count, avg_minutes_after_auto
            - detail (pd.DataFrame): Row-level dump of all qualifying after-hours manual CCs with:
                  LOAD_NUM, ENTERED_DATETIME_CST, UPDATE_USER, CUSTOMERCODE,
                  EAR2NAME, EAR2ID, preceding_auto_datetime, min_minutes_after_auto
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')

    # --- Transit window boundaries per load ---
    first_pickup = (
        df[df['CHECK_CALL_TYPE'] == 'P-Open']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .min()
        .rename('first_pickup_open')
    )
    last_dropoff = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
    )
    transit_bounds = pd.concat([first_pickup, last_dropoff], axis=1).dropna().reset_index()

    # --- After-hours manual CCs with dimension columns ---
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1) &
        (df['ROLLUPBRANCHNAME'] == branch_name)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST', 'UPDATE_USER', 'EAR2NAME', 'EAR2ID',
       'CUSTOMERCODE', 'ROLLUPBRANCHNAME']].copy()

    # Keep only manual CCs inside the transit window
    manual_cc = manual_cc.merge(transit_bounds, on='LOAD_NUM', how='inner')
    manual_cc = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] >= manual_cc['first_pickup_open']) &
        (manual_cc['ENTERED_DATETIME_CST'] <= manual_cc['last_dropoff_close'])
    ].copy()

    # --- Automated CCs (all loads, not filtered to after hours) ---
    auto_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == True) &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 0) &
        (df['UPDATE_USER'] != 'DATASCIENCE')
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].rename(
        columns={'ENTERED_DATETIME_CST': 'auto_datetime'}
    ).copy()

    # --- After-auto detection ---
    merged = manual_cc.merge(auto_cc, on='LOAD_NUM', how='left')
    merged['minutes_after_auto'] = (
        merged['ENTERED_DATETIME_CST'] - merged['auto_datetime']
    ).dt.total_seconds() / 60

    # Keep only pairings where auto CC preceded the manual CC within the window
    qualifying = merged[
        (merged['minutes_after_auto'] > 0) &
        (merged['minutes_after_auto'] <= window_minutes)
    ].copy()

    # For each manual CC, find the closest (minimum minutes) preceding auto CC
    # and capture its datetime alongside the gap
    closest_auto = (
        qualifying.sort_values('minutes_after_auto')
        .groupby(['LOAD_NUM', 'ENTERED_DATETIME_CST'], as_index=False)
        .first()[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'auto_datetime', 'minutes_after_auto']]
        .rename(columns={'minutes_after_auto': 'min_minutes_after_auto',
                         'auto_datetime': 'preceding_auto_datetime'})
    )
    closest_auto['follows_auto'] = True

    # Join back to manual CCs (preserving dimension columns)
    manual_cc = manual_cc.merge(
        closest_auto[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'follows_auto',
                      'min_minutes_after_auto', 'preceding_auto_datetime']],
        on=['LOAD_NUM', 'ENTERED_DATETIME_CST'],
        how='left'
    )
    manual_cc['follows_auto'] = manual_cc['follows_auto'].fillna(False)

    # --- Summary 1: Per-load summary ---
    load_summary = (
        manual_cc.groupby(['LOAD_NUM', 'first_pickup_open', 'last_dropoff_close'])
        .agg(
            transit_manual_cc_total=('ENTERED_DATETIME_CST', 'count'),
            manual_cc_after_auto_count=('follows_auto', 'sum'),
            avg_minutes_after_auto=('min_minutes_after_auto', 'mean')
        )
        .reset_index()
    )
    load_summary['manual_cc_after_auto_count'] = load_summary['manual_cc_after_auto_count'].astype(int)
    load_summary['avg_minutes_after_auto'] = load_summary['avg_minutes_after_auto'].round(1)
    load_summary['pct_after_auto'] = (
        load_summary['manual_cc_after_auto_count'] / load_summary['transit_manual_cc_total'] * 100
    ).round(1)
    load_summary = load_summary.sort_values('manual_cc_after_auto_count', ascending=False)

    # Keep only qualifying rows for user summary and detail
    after_auto = manual_cc[manual_cc['follows_auto']].copy()

    # --- Summary 2: By UPDATE_USER ---
    by_user = (
        after_auto.groupby('UPDATE_USER')
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            manual_cc_after_auto_count=('ENTERED_DATETIME_CST', 'count'),
            avg_minutes_after_auto=('min_minutes_after_auto', 'mean')
        )
        .reset_index()
        .sort_values('manual_cc_after_auto_count', ascending=False)
    )
    by_user['avg_minutes_after_auto'] = by_user['avg_minutes_after_auto'].round(1)

    # --- Detail: Row-level dump of all qualifying after-hours manual CCs ---
    detail = (
        after_auto[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'UPDATE_USER',
                    'CUSTOMERCODE', 'EAR2NAME', 'EAR2ID',
                    'preceding_auto_datetime', 'min_minutes_after_auto']]
        .sort_values(['LOAD_NUM', 'ENTERED_DATETIME_CST'])
        .reset_index(drop=True)
    )

    return load_summary, by_user, detail


# Run the function
ah_load_summary, ah_by_user, ah_detail = get_manual_cc_near_auto_in_transit_afterhours(df, window_minutes=30)

total_ah_transit = ah_load_summary['transit_manual_cc_total'].sum()
total_ah_after_auto = ah_load_summary['manual_cc_after_auto_count'].sum()
overall_ah_avg_minutes = ah_load_summary.loc[
    ah_load_summary['manual_cc_after_auto_count'] > 0, 'avg_minutes_after_auto'
].mean()

print(f"=== After Hours: Manual CCs Near Auto (±{30} min) — Per-Load Summary ===")
print(f"Loads with after-hours manual CCs in transit:      {len(ah_load_summary):,}")
print(f"Total after-hours manual CCs in transit:           {total_ah_transit:,}")
print(f"After-hours manual CCs within 30 min after auto:  {total_ah_after_auto:,}")
print(f"Overall % after auto CC:                           {total_ah_after_auto / total_ah_transit * 100:.1f}%" if total_ah_transit > 0 else "Overall % after auto CC: N/A")
print(f"Avg minutes after auto CC (loads with any):        {overall_ah_avg_minutes:.1f} min")
print()
display(ah_load_summary.head(10))

print("\n=== After Hours: Manual CCs Within 30 Min After Auto CC by UPDATE_USER ===")
print(f"Distinct after-hours users with after-auto touches: {len(ah_by_user):,}")
print()
display(ah_by_user.head(20))

print("\n=== After Hours: Raw After-Auto Manual CC Detail ===")
print(f"Total qualifying after-hours after-auto manual CC rows: {len(ah_detail):,}")
print()
display(ah_detail.head(20))


=== After Hours: Manual CCs Near Auto (±30 min) — Per-Load Summary ===
Loads with after-hours manual CCs in transit:      2,609
Total after-hours manual CCs in transit:           5,175
After-hours manual CCs within 30 min after auto:  3,322
Overall % after auto CC:                           64.2%
Avg minutes after auto CC (loads with any):        8.3 min



,LOAD_NUM,first_pickup_open,last_dropoff_close,transit_manual_cc_total,manual_cc_after_auto_count,avg_minutes_after_auto,pct_after_auto
181,551412317,2026-05-21 08:00:00,2026-05-27 09:00:00,19,19,9.7,100.0
1982,553916777,2026-05-21 08:00:00,2026-05-26 07:00:00,18,17,8.0,94.4
109,551098226,2026-05-01 08:00:00,2026-05-04 09:00:00,16,16,9.2,100.0
312,551733805,2026-05-01 08:00:00,2026-05-04 21:00:00,16,16,8.4,100.0
1642,553442682,2026-05-26 08:00:00,2026-06-01 05:30:00,19,16,6.6,84.2
308,551733783,2026-05-22 08:00:00,2026-05-26 11:00:00,15,15,4.6,100.0
1086,552771412,2026-05-08 08:00:00,2026-05-12 15:00:00,15,15,5.1,100.0
829,552424069,2026-05-05 06:00:00,2026-05-08 15:00:00,15,15,9.3,100.0
827,552423490,2026-05-08 13:00:00,2026-05-11 20:00:00,14,14,8.1,100.0
1839,553660696,2026-05-18 07:00:00,2026-05-20 09:00:00,14,14,9.1,100.0



=== After Hours: Manual CCs Within 30 Min After Auto CC by UPDATE_USER ===
Distinct after-hours users with after-auto touches: 117



,UPDATE_USER,distinct_loads,manual_cc_after_auto_count,avg_minutes_after_auto
79,OWENSTEP,202,208,8.0
104,TOSCARM,79,194,7.6
31,FOSTHEA,161,184,8.3
72,MOORHARO,109,180,8.7
98,STINBLAI,139,142,8.4
0,AHMABIL,135,137,8.2
81,PELARIC,27,129,7.2
112,VAZQELI,54,122,7.9
52,JOHNCH,95,118,8.9
78,ORTEPEDR,62,114,8.5



=== After Hours: Raw After-Auto Manual CC Detail ===
Total qualifying after-hours after-auto manual CC rows: 3,322



,LOAD_NUM,ENTERED_DATETIME_CST,UPDATE_USER,CUSTOMERCODE,EAR2NAME,EAR2ID,preceding_auto_datetime,min_minutes_after_auto
0,546748574,2026-05-15 23:08:00,TOSCARM,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-15 23:02:00,6.0
1,546748574,2026-05-16 03:31:00,VAZQELI,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-16 03:18:00,13.0
2,546748574,2026-05-16 08:45:00,VAZQELI,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-16 08:39:00,6.0
3,546748574,2026-05-16 14:02:00,ALTAELI,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-16 13:59:00,3.0
4,546748574,2026-05-16 19:26:00,TOSCARM,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-16 19:15:00,11.0
5,546748574,2026-05-16 23:34:00,TOSCARM,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-16 23:31:00,3.0
6,546748574,2026-05-17 02:56:00,SOTOALEJ,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-17 02:41:00,15.0
7,546748574,2026-05-17 06:05:00,SOTOALEJ,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-17 05:53:00,12.0
8,546748574,2026-05-17 11:20:00,VAZQELI,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-17 11:09:00,11.0
9,546748574,2026-05-17 15:28:00,VAZQELI,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0,2026-05-17 15:22:00,6.0


In [38]:
# Export after-hours after-auto CC summaries to Excel
output_path = f'../data/processed/afterhours_after_auto_cc_summary_{startdate}_to_{enddate}.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    ah_load_summary.to_excel(writer, sheet_name='By_Load', index=False)
    ah_by_user.to_excel(writer, sheet_name='By_User', index=False)
    ah_detail.to_excel(writer, sheet_name='Detail', index=False)

print(f"✓ Excel file saved to: {output_path}")
print(f"  Tab 'By_Load':   {len(ah_load_summary):,} rows")
print(f"  Tab 'By_User':   {len(ah_by_user):,} rows")
print(f"  Tab 'Detail':    {len(ah_detail):,} rows")


✓ Excel file saved to: ../data/processed/afterhours_after_auto_cc_summary_2026-05-01_to_2026-05-31.xlsx
  Tab 'By_Load':   2,609 rows
  Tab 'By_User':   117 rows
  Tab 'Detail':    3,322 rows


In [ ]:
def get_after_auto_cc_summary(df, window_minutes=30):
    """
    Summarizes manual check calls that occurred within `window_minutes` AFTER
    an automated CC during active transit (between first P-Open and last D-Close)
    at three aggregated levels, plus a raw row-level detail table:
      1. EAR2NAME — to identify which customer segments drive the most redundant touches
      2. UPDATE_USER + ROLLUPBRANCHNAME — to identify which dispatcher/branch combos
         are entering the most redundant manual CCs after automation
      3. UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME — to identify which user/branch/customer
         segment combos are driving the most redundant manual CCs
      4. Raw detail — one row per qualifying manual CC with load/user/customer context

    Uses the same "after auto" logic as get_manual_cc_near_auto_in_transit:
        - Manual CC must fall between first P-Open and last D-Close
        - At least one auto CC on the same load occurred within `window_minutes` BEFORE the manual CC

    Criteria for a manual check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1

    Criteria for an automated check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == True
        - HUMAN_ENTERED_CHECKCALL_FLAG == 0
        - UPDATE_USER != 'DATASCIENCE'

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        window_minutes (int): Look-back window in minutes. Default 30.

    Returns:
        tuple:
            - by_ear2 (pd.DataFrame): Aggregated by EAR2NAME
            - by_user_branch (pd.DataFrame): Aggregated by UPDATE_USER + ROLLUPBRANCHNAME
            - by_user_branch_ear2 (pd.DataFrame): Aggregated by UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME
            - detail (pd.DataFrame): Row-level dump of all qualifying manual CCs with
                                     LOAD_NUM, ENTERED_DATETIME_CST, UPDATE_USER,
                                     CUSTOMERCODE, EAR2NAME, EAR2ID, min_minutes_after_auto
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')

    # --- Transit window boundaries per load ---
    first_pickup = (
        df[df['CHECK_CALL_TYPE'] == 'P-Open']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .min()
        .rename('first_pickup_open')
    )
    last_dropoff = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
    )
    transit_bounds = pd.concat([first_pickup, last_dropoff], axis=1).dropna().reset_index()

    # --- Manual CCs with dimension columns ---
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST', 'UPDATE_USER', 'EAR2NAME', 'EAR2ID',
       'CUSTOMERCODE', 'ROLLUPBRANCHNAME']].copy()

    # Keep only manual CCs inside the transit window
    manual_cc = manual_cc.merge(transit_bounds, on='LOAD_NUM', how='inner')
    manual_cc = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] >= manual_cc['first_pickup_open']) &
        (manual_cc['ENTERED_DATETIME_CST'] <= manual_cc['last_dropoff_close'])
    ].copy()

    # --- Automated CCs ---
    auto_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == True) &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 0) &
        (df['UPDATE_USER'] != 'DATASCIENCE')
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].rename(
        columns={'ENTERED_DATETIME_CST': 'auto_datetime'}
    ).copy()

    # --- After-auto detection ---
    merged = manual_cc.merge(auto_cc, on='LOAD_NUM', how='left')
    merged['minutes_after_auto'] = (
        merged['ENTERED_DATETIME_CST'] - merged['auto_datetime']
    ).dt.total_seconds() / 60

    # Keep only pairings where auto CC preceded the manual CC within the window
    qualifying = merged[
        (merged['minutes_after_auto'] > 0) &
        (merged['minutes_after_auto'] <= window_minutes)
    ].copy()

    # For each manual CC, use the closest (minimum minutes) preceding auto CC
    closest_auto = (
        qualifying.groupby(['LOAD_NUM', 'ENTERED_DATETIME_CST'])['minutes_after_auto']
        .min()
        .reset_index()
        .rename(columns={'minutes_after_auto': 'min_minutes_after_auto'})
    )
    closest_auto['follows_auto'] = True

    # Join back to manual CCs (preserving dimension columns)
    manual_cc = manual_cc.merge(
        closest_auto[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'follows_auto', 'min_minutes_after_auto']],
        on=['LOAD_NUM', 'ENTERED_DATETIME_CST'],
        how='left'
    )
    manual_cc['follows_auto'] = manual_cc['follows_auto'].fillna(False)

    # Keep only qualifying rows for the summaries
    after_auto = manual_cc[manual_cc['follows_auto']].copy()

    # --- Summary 1: By EAR2NAME ---
    by_ear2 = (
        after_auto.groupby('EAR2NAME')
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            manual_cc_after_auto_count=('ENTERED_DATETIME_CST', 'count'),
            avg_minutes_after_auto=('min_minutes_after_auto', 'mean')
        )
        .reset_index()
        .sort_values('manual_cc_after_auto_count', ascending=False)
    )
    by_ear2['avg_minutes_after_auto'] = by_ear2['avg_minutes_after_auto'].round(1)

    # --- Summary 2: By UPDATE_USER + ROLLUPBRANCHNAME ---
    by_user_branch = (
        after_auto.groupby(['UPDATE_USER', 'ROLLUPBRANCHNAME'])
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            manual_cc_after_auto_count=('ENTERED_DATETIME_CST', 'count'),
            avg_minutes_after_auto=('min_minutes_after_auto', 'mean')
        )
        .reset_index()
        .sort_values('manual_cc_after_auto_count', ascending=False)
    )
    by_user_branch['avg_minutes_after_auto'] = by_user_branch['avg_minutes_after_auto'].round(1)

    # --- Summary 3: By UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME ---
    by_user_branch_ear2 = (
        after_auto.groupby(['UPDATE_USER', 'ROLLUPBRANCHNAME', 'EAR2NAME'])
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            manual_cc_after_auto_count=('ENTERED_DATETIME_CST', 'count'),
            avg_minutes_after_auto=('min_minutes_after_auto', 'mean')
        )
        .reset_index()
        .sort_values('manual_cc_after_auto_count', ascending=False)
    )
    by_user_branch_ear2['avg_minutes_after_auto'] = by_user_branch_ear2['avg_minutes_after_auto'].round(1)

    # --- Detail: Row-level dump of all qualifying manual CCs ---
    detail = (
        after_auto[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'UPDATE_USER',
                    'CUSTOMERCODE', 'EAR2NAME', 'EAR2ID', 'min_minutes_after_auto']]
        .sort_values(['LOAD_NUM', 'ENTERED_DATETIME_CST'])
        .reset_index(drop=True)
    )

    return by_ear2, by_user_branch, by_user_branch_ear2, detail


# Run the function
after_auto_by_ear2, after_auto_by_user_branch, after_auto_by_user_branch_ear2, after_auto_detail = get_after_auto_cc_summary(df, window_minutes=30)

print("=== Manual CCs Within 30 Min After Auto CC by EAR2NAME ===")
print(f"Distinct EAR2s with after-auto touches: {len(after_auto_by_ear2):,}")
print()
display(after_auto_by_ear2.head(15))

print("\n=== Manual CCs Within 30 Min After Auto CC by UPDATE_USER / ROLLUPBRANCHNAME ===")
print(f"Distinct user/branch combos with after-auto touches: {len(after_auto_by_user_branch):,}")
print()
display(after_auto_by_user_branch.head(20))

print("\n=== Manual CCs Within 30 Min After Auto CC by UPDATE_USER / ROLLUPBRANCHNAME / EAR2NAME ===")
print(f"Distinct user/branch/EAR2 combos with after-auto touches: {len(after_auto_by_user_branch_ear2):,}")
print()
display(after_auto_by_user_branch_ear2.head(20))

print("\n=== Raw After-Auto Manual CC Detail ===")
print(f"Total qualifying after-auto manual CC rows: {len(after_auto_detail):,}")
print()
display(after_auto_detail.head(20))


=== Manual CCs Sandwiched by Auto CCs (±30 min) by EAR2NAME ===
Distinct EAR2s with sandwiched touches: 1,447



,EAR2NAME,distinct_loads,manual_cc_sandwiched_count
481,"Fábrica de Jabón La Corona, S.A. de C.V.",101,723
168,BorgWarner Inc.,113,411
644,International Paper Company,306,373
385,Eaton Corporation,42,177
981,Phillips 66,76,157
1087,Robinson Fresh,123,135
1255,The Coca-Cola Company,120,128
852,Molson Coors Beverage Company,92,102
1022,ProAmpac Holdings Inc.,80,86
1240,"Tesla, Inc.",75,84



=== Manual CCs Sandwiched by Auto CCs (±30 min) by UPDATE_USER / ROLLUPBRANCHNAME ===
Distinct user/branch combos with sandwiched touches: 586



,UPDATE_USER,ROLLUPBRANCHNAME,distinct_loads,manual_cc_sandwiched_count
88,CASTAN,Dallas Capacity,497,592
526,TOSCARM,NAST After Hours,78,186
393,OWENSTEP,NAST After Hours,182,186
159,FOSTHEA,NAST After Hours,151,174
358,MOORHARO,NAST After Hours,104,161
535,TURNJASM,Memphis IP,89,137
166,GARZNES,Carrier Support,124,135
508,STINBLAI,NAST After Hours,126,129
401,PELARIC,NAST After Hours,27,126
2,AHMABIL,NAST After Hours,123,125



=== Manual CCs Sandwiched by Auto CCs (±30 min) by UPDATE_USER / ROLLUPBRANCHNAME / EAR2NAME ===
Distinct user/branch/EAR2 combos with sandwiched touches: 4,894



,UPDATE_USER,ROLLUPBRANCHNAME,EAR2NAME,distinct_loads,manual_cc_sandwiched_count
4559,TOSCARM,NAST After Hours,"Fábrica de Jabón La Corona, S.A. de C.V.",77,184
4625,TURNJASM,Memphis IP,International Paper Company,89,137
4672,VAZQELI,NAST After Hours,"Fábrica de Jabón La Corona, S.A. de C.V.",46,107
3540,PELARIC,NAST After Hours,BorgWarner Inc.,19,105
3398,ORTEPEDR,NAST After Hours,"Fábrica de Jabón La Corona, S.A. de C.V.",54,97
1978,HERFER,NAST After Hours,"Fábrica de Jabón La Corona, S.A. de C.V.",47,97
3241,MOORHARO,NAST After Hours,Phillips 66,43,96
2848,LOPEPAT1,NAST After Hours,BorgWarner Inc.,27,72
1165,CRUZMAN,NAST After Hours,BorgWarner Inc.,19,68
1465,FOSTHEA,NAST After Hours,BorgWarner Inc.,48,64



=== Raw Sandwiched Manual CC Detail ===
Total qualifying sandwiched manual CC rows: 8,715



,LOAD_NUM,ENTERED_DATETIME_CST,UPDATE_USER,CUSTOMERCODE,EAR2NAME,EAR2ID
0,540066491,2026-05-05 06:53:00,KAPLRUS,C8897795,Trader Joe's Company,67.0
1,543505728,2026-05-18 07:14:00,KUNIAND,C671020,CCI Manufacturing IL Corporation,297137.0
2,544683700,2026-05-01 08:13:00,CASTAN,C7348876,Generac Holdings Inc.,15639.0
3,544683700,2026-05-01 12:27:00,CASTAN,C7348876,Generac Holdings Inc.,15639.0
4,544683742,2026-05-19 12:40:00,RODRTHO,C7348876,Generac Holdings Inc.,15639.0
5,544742538,2026-05-20 06:43:00,CLUMJOS,C8029570,"GC Ingredients, Inc",189139.0
6,544867969,2026-05-15 11:42:00,ARENMAT,C1081443,"LCT OpCo, LLC",35225.0
7,544980290,2026-05-28 09:00:00,JACOWIL,C8271441,"Unilever Manufacturing (US), Inc.",1056913.0
8,545173583,2026-05-28 10:44:00,BERGJAM,C8012738,Ruxton Chocolates LLC,28865.0
9,545181989,2026-05-21 06:24:00,KALMROS,C7043477,"Hyland Filter Service, Inc.",445020.0


In [ ]:
# # Export after-auto CC summaries to Excel
# output_path = f'../data/processed/after_auto_cc_summary_{startdate}_to_{enddate}_updated.xlsx'

# with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
#     after_auto_by_ear2.to_excel(writer, sheet_name='By_EAR2NAME', index=False)
#     after_auto_by_user_branch.to_excel(writer, sheet_name='By_User_Branch', index=False)
#     after_auto_by_user_branch_ear2.to_excel(writer, sheet_name='By_User_Branch_EAR2', index=False)
#     after_auto_detail.to_excel(writer, sheet_name='Detail', index=False)

# print(f"✓ Excel file saved to: {output_path}")
# print(f"  Tab 'By_EAR2NAME':           {len(after_auto_by_ear2):,} rows")
# print(f"  Tab 'By_User_Branch':        {len(after_auto_by_user_branch):,} rows")
# print(f"  Tab 'By_User_Branch_EAR2':   {len(after_auto_by_user_branch_ear2):,} rows")
# print(f"  Tab 'Detail':                {len(after_auto_detail):,} rows")


✓ Excel file saved to: ../data/processed/sandwiched_cc_summary_2026-05-01_to_2026-05-31.xlsx
  Tab 'By_EAR2NAME':           1,402 rows
  Tab 'By_User_Branch':        575 rows
  Tab 'By_User_Branch_EAR2':   4,692 rows
  Tab 'Detail':                8,340 rows


In [20]:
def get_loads_with_manual_cc_after_dropoff(df):
    """
    Identifies LOAD_NUMs that have at least one manual check call
    (CHECK_CALL_TYPE = 'CC' and AUTOMATED = False) recorded AFTER
    the last drop off close event (CHECK_CALL_TYPE = 'D-Close').
    Excludes check calls where UPDATE_USER = 'DATASCIENCE'.
    Only includes check calls where HUMAN_ENTERED_CHECKCALL_FLAG = 1.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake containing columns:
                           LOAD_NUM, CHECK_CALL_TYPE, AUTOMATED, ENTERED_DATETIME_CST,
                           UPDATE_USER, HUMAN_ENTERED_CHECKCALL_FLAG

    Returns:
        pd.DataFrame: DataFrame of qualifying LOAD_NUMs with supporting details:
                      - LOAD_NUM
                      - last_dropoff_close: latest D-Close datetime
                      - manual_cc_after_dropoff_count: number of manual CCs after D-Close
                      - latest_manual_cc: datetime of the last manual CC after D-Close
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')

    # Get the last D-Close datetime per load
    dropoff_close = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
        .reset_index()
    )

    # Get manual check calls (CC + not automated + not DATASCIENCE user + human entered)
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Join manual CCs with their load's last D-Close datetime
    manual_cc = manual_cc.merge(dropoff_close, on='LOAD_NUM', how='inner')

    # Keep only manual CCs that occurred AFTER the last D-Close
    manual_cc_after = manual_cc[
        manual_cc['ENTERED_DATETIME_CST'] > manual_cc['last_dropoff_close']
    ]

    # Aggregate per load
    result = (
        manual_cc_after.groupby(['LOAD_NUM', 'last_dropoff_close'])
        .agg(
            manual_cc_after_dropoff_count=('ENTERED_DATETIME_CST', 'count'),
            latest_manual_cc=('ENTERED_DATETIME_CST', 'max')
        )
        .reset_index()
        .sort_values('manual_cc_after_dropoff_count', ascending=False)
    )

    return result


# Run the function
loads_with_late_manual_cc = get_loads_with_manual_cc_after_dropoff(df)
qualifying_loads_after = len(loads_with_late_manual_cc)

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"LOAD_NUMs with manual CC after drop off close:     {qualifying_loads_after:,}")
print(f"Percentage:                                        {qualifying_loads_after / total_loads * 100:.1f}%")
print(f"Total Manual CCs after Drop Off Close:             {loads_with_late_manual_cc['manual_cc_after_dropoff_count'].sum():,}")
loads_with_late_manual_cc.head(10)


Total distinct LOAD_NUMs in dataset:               311,359
LOAD_NUMs with manual CC after drop off close:     857
Percentage:                                        0.3%
Total Manual CCs after Drop Off Close:             991


,LOAD_NUM,last_dropoff_close,manual_cc_after_dropoff_count,latest_manual_cc
619,553654474,2026-05-23 09:00:00,9,2026-05-23 14:31:00
730,554146525,2026-05-23 14:00:00,9,2026-05-25 07:30:00
395,552901255,2026-05-13 15:00:00,8,2026-05-15 06:18:00
789,554495801,2026-05-27 12:00:00,7,2026-05-29 06:25:00
378,552859642,2026-05-09 02:00:00,7,2026-05-10 09:58:00
207,552282398,2026-05-05 17:00:00,7,2026-05-06 07:35:00
747,554260994,2026-05-25 11:00:00,6,2026-05-28 04:37:00
143,552046963,2026-05-07 21:00:00,6,2026-05-08 08:08:00
166,552153035,2026-05-06 17:00:00,6,2026-05-07 08:11:00
182,552196199,2026-05-05 11:00:00,5,2026-05-06 08:18:00


In [21]:
def get_manual_cc_after_last_auto(df):
    """
    Identifies per load how many manual check calls occurred AFTER the last
    automated check call AND BEFORE the last D-Close.

    These represent manual touches that were required because automated tracking
    stopped firing before the load was delivered — an opportunity to reduce
    over-processing by improving automation continuity.

    Criteria for a manual check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1

    Criteria for an automated check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == True
        - HUMAN_ENTERED_CHECKCALL_FLAG == 0
        - UPDATE_USER == 'DATASCIENCE'

    Window of interest per load:
        - Start: last automated CC datetime (exclusive)
        - End:   last D-Close datetime (inclusive)

    Only loads that have both an automated CC and a D-Close are included.
    Loads where the last auto CC is already after the last D-Close are excluded.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.

    Returns:
        pd.DataFrame: One row per qualifying load with:
            - LOAD_NUM
            - last_auto_cc: datetime of the last automated CC on the load
            - last_dropoff_close: datetime of the last D-Close on the load
            - auto_gap_minutes: minutes between last auto CC and last D-Close
            - manual_cc_in_gap_count: manual CCs in the gap (after last auto, before D-Close)
            - earliest_gap_manual_cc: datetime of the first manual CC in the gap
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')

    # --- Last automated CC per load ---
    last_auto = (
        df[
            (df['CHECK_CALL_TYPE'] == 'CC') &
            (df['AUTOMATED'] == True) &
            (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 0) &
            (df['UPDATE_USER'] != 'DATASCIENCE')
        ]
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_auto_cc')
        .reset_index()
    )

    # --- Last D-Close per load ---
    last_dropoff = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
        .reset_index()
    )

    # --- Combine: only loads where last auto CC is BEFORE last D-Close ---
    bounds = last_auto.merge(last_dropoff, on='LOAD_NUM', how='inner')
    bounds = bounds[bounds['last_auto_cc'] < bounds['last_dropoff_close']].copy()
    bounds['auto_gap_minutes'] = (
        (bounds['last_dropoff_close'] - bounds['last_auto_cc'])
        .dt.total_seconds() / 60
    ).round(1)

    # --- Manual CCs ---
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Keep only manual CCs that fall strictly after the last auto CC and before/at D-Close
    manual_cc = manual_cc.merge(bounds, on='LOAD_NUM', how='inner')
    manual_cc_in_gap = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] > manual_cc['last_auto_cc']) &
        (manual_cc['ENTERED_DATETIME_CST'] <= manual_cc['last_dropoff_close'])
    ]

    # --- Aggregate per load ---
    result = (
        manual_cc_in_gap.groupby(['LOAD_NUM', 'last_auto_cc', 'last_dropoff_close', 'auto_gap_minutes'])
        .agg(
            manual_cc_in_gap_count=('ENTERED_DATETIME_CST', 'count'),
            earliest_gap_manual_cc=('ENTERED_DATETIME_CST', 'min')
        )
        .reset_index()
        .sort_values('manual_cc_in_gap_count', ascending=False)
    )

    return result, bounds


# Run the function
manual_cc_after_last_auto, gap_bounds = get_manual_cc_after_last_auto(df)
loads_with_auto_gap = len(manual_cc_after_last_auto)
total_gap_manual = manual_cc_after_last_auto['manual_cc_in_gap_count'].sum()
avg_gap_minutes = gap_bounds['auto_gap_minutes'].median()

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"Loads where last auto CC precedes D-Close:         {len(gap_bounds):,}")
print(f"Loads with manual CCs filling the auto gap:        {loads_with_auto_gap:,}")
print(f"  % of gap loads with manual fill-in:              {loads_with_auto_gap / len(gap_bounds) * 100:.1f}%")
print(f"Total manual CCs in auto gap:                      {total_gap_manual:,}")
print(f"Median gap duration (last auto CC → D-Close):      {avg_gap_minutes:.0f} min")
manual_cc_after_last_auto.head(10)


Total distinct LOAD_NUMs in dataset:               311,359
Loads where last auto CC precedes D-Close:         181,792
Loads with manual CCs filling the auto gap:        991
  % of gap loads with manual fill-in:              0.5%
Total manual CCs in auto gap:                      1,258
Median gap duration (last auto CC → D-Close):      254 min


,LOAD_NUM,last_auto_cc,last_dropoff_close,auto_gap_minutes,manual_cc_in_gap_count,earliest_gap_manual_cc
302,552313222,2026-05-04 15:19:00,2026-05-07 18:00:00,4481.0,19,2026-05-04 18:31:00
939,554871037,2026-05-26 12:26:00,2026-05-28 18:00:00,3214.0,13,2026-05-26 18:45:00
600,553360137,2026-05-14 17:29:00,2026-05-15 17:00:00,1411.0,12,2026-05-14 19:21:00
320,552404766,2026-05-04 17:51:00,2026-05-06 11:00:00,2469.0,11,2026-05-04 22:30:00
261,552223612,2026-05-13 17:41:00,2026-05-18 10:30:00,6769.0,8,2026-05-13 23:53:00
355,552527278,2026-05-05 12:59:00,2026-05-06 16:00:00,1621.0,6,2026-05-05 18:58:00
879,554311761,2026-05-24 07:30:00,2026-05-26 10:00:00,3030.0,6,2026-05-24 14:47:00
23,550784132,2026-05-02 19:09:00,2026-05-04 12:00:00,2451.0,5,2026-05-02 21:06:00
617,553433976,2026-05-16 20:14:00,2026-05-18 14:00:00,2506.0,5,2026-05-16 21:54:00
319,552404168,2026-05-04 15:35:00,2026-05-08 15:00:00,5725.0,5,2026-05-05 11:15:00


In [22]:
df[(df['LOAD_NUM']==546748574) & (pd.isnull(df['JOB_FAMILY_DESCRIPTION'])==False)][['LOAD_NUM','EAR2NAME','DESCRIPTION','ENTERED_DATETIME_CST','UPDATE_USER']]

,LOAD_NUM,EAR2NAME,DESCRIPTION,ENTERED_DATETIME_CST,UPDATE_USER
21134021,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-15 23:08:00,TOSCARM
21369913,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-16 03:31:00,VAZQELI
21649884,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-16 08:45:00,VAZQELI
21951463,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-16 14:02:00,ALTAELI
22245156,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-16 19:26:00,TOSCARM
22451720,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-16 23:34:00,TOSCARM
22608074,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 02:56:00,SOTOALEJ
22749745,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 06:05:00,SOTOALEJ
23001225,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 11:20:00,VAZQELI
23211180,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 15:28:00,VAZQELI
